In [1]:
!pip install pandas numpy yfinance ta matplotlib

In [2]:
# =========================
# 1. IMPORTS
# =========================

import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt

In [3]:
# =========================
# 2. INDICATORS
# =========================

def add_emas(df):
    df = df.copy()
    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()
    df["EMA50"] = df["Close"].ewm(span=50).mean()
    return df


def add_atr(df, window=14):
    df = df.copy()
    df["ATR"] = ta.volatility.average_true_range(
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        window=window
    )
    return df

In [4]:
# =========================
# 3. DATA DOWNLOAD
# =========================

def clean_columns(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def download_data(symbol):
    daily = yf.download(symbol, period="2y", interval="1d", auto_adjust=False, progress=False)
    hourly = yf.download(symbol, period="60d", interval="1h", auto_adjust=False, progress=False)

    daily = clean_columns(daily)
    hourly = clean_columns(hourly)

    daily = add_emas(daily)
    hourly = add_emas(hourly)
    hourly = add_atr(hourly)

    return daily, hourly

In [5]:
# =========================
# 4. TIMEFRAME BUILDING
# =========================

def build_weekly_monthly(daily):
    weekly = daily.resample("W").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    monthly = daily.resample("ME").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    weekly = add_emas(weekly)
    monthly = add_emas(monthly)

    return weekly, monthly

In [6]:
# =========================
# 5. 100-DAY HIGH RULE
# =========================

def hundred_day_high_recent(daily, lookback_days=100, recent_days=8):
    if len(daily) < lookback_days:
        return False

    current_100_day_high = daily["High"].rolling(lookback_days).max().iloc[-1]
    recent_high = daily["High"].tail(recent_days).max()

    return recent_high >= current_100_day_high

In [17]:
# =========================
# 6. FILTERS AT A HISTORICAL TIME
# =========================

def check_filters_at_time(daily, hourly, current_time):
    current_time_clean = pd.Timestamp(current_time).tz_localize(None)
    current_date = current_time_clean.normalize()

    daily_clean = daily.copy()
    hourly_clean = hourly.copy()

    daily_clean.index = daily_clean.index.tz_localize(None)
    hourly_clean.index = hourly_clean.index.tz_localize(None)

    daily_past = daily_clean.loc[:current_date].iloc[:-1].copy()
    hourly_past = hourly_clean.loc[:current_time_clean].copy()

    if len(daily_past) < 100 or len(hourly_past) < 50:
        return {
            "monthly": False,
            "weekly": False,
            "daily": False,
            "hourly": False,
            "100D": False,
            "overall": False
        }

    weekly, monthly = build_weekly_monthly(daily_past)

    if len(monthly) < 12 or len(weekly) < 20:
        return {
            "monthly": False,
            "weekly": False,
            "daily": False,
            "hourly": False,
            "100D": False,
            "overall": False
        }

    last_month = monthly.iloc[-1]
    last_week = weekly.iloc[-1]
    last_day = daily_past.iloc[-1]
    recent_20_hours = hourly_past.tail(20)
    recent_15_hours = hourly_past.tail(15)

    monthly_pass = (
        last_month["Close"] > last_month["EMA9"]
        and last_month["Close"] > last_month["EMA20"]
        and last_month["Close"] > last_month["EMA50"]
    )

    weekly_pass = (
        last_week["Close"] > last_week["EMA9"]
        and last_week["Close"] > last_week["EMA20"]
        and last_week["Close"] > last_week["EMA50"]
    )

    daily_pass = (
        last_day["Close"] > last_day["EMA20"]
        and last_day["Close"] > last_day["EMA50"]
    )

    closed_below_50_recently = (
        recent_20_hours["Close"] < recent_20_hours["EMA50"]
    ).any()

    crossed_above_20_recently = (
        (recent_15_hours["Close"] > recent_15_hours["EMA20"])
        & (recent_15_hours["Close"].shift(1) <= recent_15_hours["EMA20"].shift(1))
    ).any()

    hundred_day_pass = hundred_day_high_recent(daily_past)

    hourly_pass = closed_below_50_recently and crossed_above_20_recently

    overall = (
        monthly_pass
        and weekly_pass
        and daily_pass
        and hourly_pass
        and hundred_day_pass
    )

    return {
        "monthly": monthly_pass,
        "weekly": weekly_pass,
        "daily": daily_pass,
        "hourly": hourly_pass,
        "100D": hundred_day_pass,
        "overall": overall
    }

In [65]:
# =========================
# 7. BACKTEST STRATEGY

def backtest_hourly_strategy(symbol):
    daily, hourly = download_data(symbol)

    trades = []

    trade_taken_dates = set()

    locked = False
    lock_high = None

    order_active = False
    in_trade = False

    signal_high = None
    signal_time = None
    signal_atr = None
    order_created_index = None

    pullback_seen = False
    reclaim_seen = False

    entry_price = None
    stop_loss = None
    initial_risk = None
    entry_time = None
    moved_to_breakeven = False

    for i in range(50, len(hourly) - 10):
        current_time = hourly.index[i]
        candle = hourly.iloc[i]

        current_time_clean = pd.Timestamp(current_time).tz_localize(None)
        current_date = current_time_clean.normalize()
        trade_date = current_time_clean.date()

        daily_clean = daily.copy()
        daily_clean.index = daily_clean.index.tz_localize(None)
        daily_past = daily_clean.loc[:current_date].iloc[:-1]

        if len(daily_past) < 120:
            continue

        recent_10d_high = daily_past["High"].tail(10).max()

        # =========================
        # MANAGE OPEN TRADE
        # =========================
        if in_trade:
            current_r = candle["High"] - entry_price

            # Move to breakeven after +1R
            if not moved_to_breakeven and candle["High"] >= entry_price + initial_risk:
                stop_loss = entry_price
                moved_to_breakeven = True

            # Trail stop at 1H EMA20, but never lower stop
            if candle["EMA20"] > stop_loss:
                stop_loss = candle["EMA20"]

            # Stop hit
            if candle["Low"] <= stop_loss:
                exit_price = stop_loss
                r_result = (exit_price - entry_price) / initial_risk

                trades.append({
                    "Ticker": symbol,
                    "Signal Time": signal_time,
                    "Entry Time": entry_time,
                    "Exit Time": current_time,
                    "Entry Price": entry_price,
                    "Exit Price": exit_price,
                    "Initial Stop": entry_price - initial_risk,
                    "Final Stop": stop_loss,
                    "R Result": r_result,
                    "Moved BE": moved_to_breakeven
                })
                
                trade_taken_dates.add(trade_date)
                in_trade = False
                locked = True
                lock_high = recent_10d_high

                pullback_seen = False
                reclaim_seen = False
                continue

            continue

        # =========================
        # RESET LOCK
        # =========================
        if locked:
            if candle["High"] > lock_high:
                locked = False
                lock_high = None
                pullback_seen = False
                reclaim_seen = False
                order_active = False
            else:
                continue

        # =========================
        # MANAGE ACTIVE BUY STOP
        # =========================
        if order_active:
            candles_waited = i - order_created_index

            # Entry check first
            if candle["High"] > signal_high:
                entry_price = signal_high
                entry_time = current_time
                initial_risk = signal_atr
                stop_loss = entry_price - initial_risk
                moved_to_breakeven = False

                in_trade = True
                order_active = False
                trade_taken_dates.add(trade_date)
                continue

            # Cancel if drops 2 ATR below setup high
            if candle["Low"] <= signal_high - (2 * signal_atr):
                order_active = False
                signal_high = None
                signal_time = None
                signal_atr = None
                order_created_index = None
                pullback_seen = False
                reclaim_seen = False
                continue

            # Expires after 8 candles
            if candles_waited > 8:
                order_active = False
                signal_high = None
                signal_time = None
                signal_atr = None
                order_created_index = None
                pullback_seen = False
                reclaim_seen = False
                continue

            continue

        # =========================
        # CHECK FILTERS
        # =========================
        filters = check_filters_at_time(daily, hourly, current_time)

        if not (
            filters["monthly"]
            and filters["weekly"]
            and filters["daily"]
            and filters["100D"]
        ):
            pullback_seen = False
            reclaim_seen = False
            continue

        # =========================
        # HOURLY SETUP
        # =========================

        if not pullback_seen:
            if candle["Close"] < candle["EMA50"]:
                pullback_seen = True
            continue

        if pullback_seen and not reclaim_seen:
            if candle["Close"] > candle["EMA20"]:
                reclaim_seen = True
            continue

        if reclaim_seen:
            next_candle = hourly.iloc[i + 1]

            # Max 1 trade per day
            if trade_date in trade_taken_dates:
                continue

            if candle["Close"] <= candle["EMA20"]:
                continue

            # Signal candle must be bullish
            if candle["Close"] <= candle["Open"]:
                continue

            if next_candle["High"] >= candle["High"]:
                continue

            # Do not create setup above/breaking recent 10D high
            if candle["High"] >= recent_10d_high or candle["Open"] >= recent_10d_high:
                locked = True
                lock_high = recent_10d_high
                pullback_seen = False
                reclaim_seen = False
                continue

            signal_high = candle["High"]
            signal_time = current_time
            signal_atr = candle["ATR"]
            order_created_index = i
            order_active = True

            continue

    return pd.DataFrame(trades)

In [71]:
# =========================
# 8. TEST IT
# =========================

trades = backtest_hourly_strategy("NOK")
trades

,Ticker,Signal Time,Entry Time,Exit Time,Entry Price,Exit Price,Initial Stop,Final Stop,R Result,Moved BE
0,NOK,2026-05-08 10:30:00-04:00,2026-05-11 09:30:00-04:00,2026-05-11 10:30:00-04:00,13.038500,13.038500,12.769129,13.038500,0.000000,True
1,NOK,2026-05-19 13:30:00-04:00,2026-05-20 09:30:00-04:00,2026-05-20 10:30:00-04:00,13.990000,13.723065,13.695735,13.723065,-0.907124,False
2,NOK,2026-06-01 14:30:00-04:00,2026-06-02 09:30:00-04:00,2026-06-03 09:30:00-04:00,16.459999,16.459999,16.173623,16.459999,0.000000,True


In [66]:
# =========================
# 9. TEST MULTIPLE TICKERS
# =========================

tickers = ["CVS", "AAPL", "NVDA", "MSFT", "META", "AMD", "TSLA", "STX", "NOK"]

for ticker in tickers:
    trades = backtest_hourly_strategy(ticker)
    print(ticker, "trades found:", len(trades))

CVS trades found: 0
AAPL trades found: 0
NVDA trades found: 1
MSFT trades found: 0
META trades found: 0
AMD trades found: 2
TSLA trades found: 0
STX trades found: 1
NOK trades found: 3


In [67]:
def backtest_stats(trades):
    if len(trades) == 0:
        return {
            "Trades": 0,
            "Wins": 0,
            "Losses": 0,
            "Breakevens": 0,
            "Win Rate": None,
            "Average R": None,
            "Total R": 0,
            "Best R": None,
            "Worst R": None
        }

    wins = trades[trades["R Result"] > 0]
    losses = trades[trades["R Result"] < 0]
    breakevens = trades[trades["R Result"] == 0]

    return {
        "Trades": len(trades),
        "Wins": len(wins),
        "Losses": len(losses),
        "Breakevens": len(breakevens),
        "Win Rate": len(wins) / len(trades),
        "Average R": trades["R Result"].mean(),
        "Total R": trades["R Result"].sum(),
        "Best R": trades["R Result"].max(),
        "Worst R": trades["R Result"].min()
    }

In [73]:
test_10 = ["VSH", "NOK", "STX", "SNDK", "META", "MCHP", "WDC", "AVGO", "WOLF", "NFLX"]

In [74]:
all_trades = []

for ticker in test_10:
    trades = backtest_hourly_strategy(ticker)
    print(ticker, "trades:", len(trades))
    if len(trades) > 0:
        all_trades.append(trades)

if len(all_trades) > 0:
    all_trades = pd.concat(all_trades, ignore_index=True)
else:
    all_trades = pd.DataFrame()

all_trades

VSH trades: 2
NOK trades: 3
STX trades: 1
SNDK trades: 2
META trades: 0
MCHP trades: 1
WDC trades: 1
AVGO trades: 1
WOLF trades: 0
NFLX trades: 0


,Ticker,Signal Time,Entry Time,Exit Time,Entry Price,Exit Price,Initial Stop,Final Stop,R Result,Moved BE
0,VSH,2026-05-20 11:30:00-04:00,2026-05-20 13:30:00-04:00,2026-05-29 13:30:00-04:00,39.660000,50.906131,38.650983,50.906131,11.145632,True
1,VSH,2026-06-11 10:30:00-04:00,2026-06-11 12:30:00-04:00,2026-06-12 09:30:00-04:00,58.480000,57.543136,56.079327,57.543136,-0.390250,False
2,NOK,2026-05-08 10:30:00-04:00,2026-05-11 09:30:00-04:00,2026-05-11 10:30:00-04:00,13.038500,13.038500,12.769129,13.038500,0.000000,True
3,NOK,2026-05-19 13:30:00-04:00,2026-05-20 09:30:00-04:00,2026-05-20 10:30:00-04:00,13.990000,13.723065,13.695735,13.723065,-0.907124,False
4,NOK,2026-06-01 14:30:00-04:00,2026-06-02 09:30:00-04:00,2026-06-03 09:30:00-04:00,16.459999,16.459999,16.173623,16.459999,0.000000,True
5,STX,2026-05-20 10:30:00-04:00,2026-05-21 09:30:00-04:00,2026-05-29 13:30:00-04:00,762.830017,874.286433,744.840723,874.286433,6.195708,True
6,SNDK,2026-05-20 10:30:00-04:00,2026-05-21 09:30:00-04:00,2026-05-27 09:30:00-04:00,1421.229980,1539.292842,1379.454273,1539.292842,2.826113,True
7,SNDK,2026-06-10 13:30:00-04:00,2026-06-11 09:30:00-04:00,2026-06-16 11:30:00-04:00,1682.890015,2001.402147,1617.332540,2001.402147,4.858517,True
8,MCHP,2026-05-13 11:30:00-04:00,2026-05-13 13:30:00-04:00,2026-05-13 14:30:00-04:00,97.980003,97.299771,96.277021,97.299771,-0.399436,False
9,WDC,2026-05-20 10:30:00-04:00,2026-05-21 09:30:00-04:00,2026-05-29 09:30:00-04:00,469.399994,528.012012,458.727684,528.012012,5.491971,True


In [72]:
backtest_stats(all_trades)

{'Trades': 13,
 'Wins': 5,
 'Losses': 6,
 'Breakevens': 2,
 'Win Rate': 0.38461538461538464,
 'Average R': np.float64(1.9861242603972675),
 'Total R': np.float64(25.81961538516448),
 'Best R': 11.145631857070713,
 'Worst R': -1.000000000000004}

In [81]:
# =========================
# 10. SCANNER
# =========================

def scan_ticker(symbol, benchmark="QQQ"):
    daily, hourly = download_data(symbol)
    qqq_daily, _ = download_data(benchmark)

    current_time = hourly.index[-1]
    filters = check_filters_at_time(daily, hourly, current_time)

    last_day = daily.iloc[-1]
    last_hour = hourly.iloc[-1]

    recent_10d_high = daily["High"].tail(10).max()
    distance_to_10d_high = (recent_10d_high - last_hour["Close"]) / last_hour["Close"]

    volume_7d_avg = daily["Volume"].tail(7).mean()

    stock_3m_return = (daily["Close"].iloc[-1] / daily["Close"].iloc[-63] - 1) * 100
    qqq_3m_return = (qqq_daily["Close"].iloc[-1] / qqq_daily["Close"].iloc[-63] - 1) * 100
    relative_strength_3m = stock_3m_return - qqq_3m_return

    trend_score = 0
    trend_score += int(filters["monthly"]) * 20
    trend_score += int(filters["weekly"]) * 20
    trend_score += int(filters["daily"]) * 20
    trend_score += int(filters["100D"]) * 20
    trend_score += int(relative_strength_3m > 0) * 20

    return {
        "Ticker": symbol,
        "Price": last_hour["Close"],
        "3M Return %": stock_3m_return,
        "QQQ 3M Return %": qqq_3m_return,
        "Relative Strength 3M %": relative_strength_3m,
        "Monthly": filters["monthly"],
        "Weekly": filters["weekly"],
        "Daily": filters["daily"],
        "100D": filters["100D"],
        "Trend Score": trend_score,
        "Distance to 10D High %": distance_to_10d_high * 100,
        "7D Avg Volume": volume_7d_avg,
        "Above 20D EMA": last_day["Close"] > last_day["EMA20"],
        "Above 50D EMA": last_day["Close"] > last_day["EMA50"],
        "Above 20H EMA": last_hour["Close"] > last_hour["EMA20"],
        "Above 50H EMA": last_hour["Close"] > last_hour["EMA50"],
    }

In [84]:
watchlist = ["AMD", "NVDA", "AAPL", "MSFT", "META", "AMZN", "GOOGL", "NFLX", "AVGO", "TSLA"]

rows = []

for ticker in watchlist:
    rows.append(scan_ticker(ticker))

scanner = pd.DataFrame(rows)

scanner.sort_values(
    by=["Relative Strength 3M %", "Trend Score"],
    ascending=[False, False]
)

,Ticker,Price,3M Return %,QQQ 3M Return %,Relative Strength 3M %,Monthly,Weekly,Daily,100D,Trend Score,Distance to 10D High %,7D Avg Volume,Above 20D EMA,Above 50D EMA,Above 20H EMA,Above 50H EMA
0,AMD,551.679993,172.167962,25.501703,146.666259,True,True,True,True,100,2.050101,3.153217e+07,True,True,True,True
8,AVGO,392.040009,21.586925,25.501703,-3.914777,True,True,True,False,60,5.764719,3.402504e+07,False,False,False,False
1,NVDA,208.559998,18.794121,25.501703,-6.707581,True,True,True,False,60,2.603571,1.482271e+08,False,True,False,False
2,AAPL,296.790009,18.100125,25.501703,-7.401578,True,True,False,False,40,6.944299,4.863453e+07,False,True,False,False
6,GOOGL,349.489990,15.765078,25.501703,-9.736624,True,True,False,False,40,7.585342,3.350063e+07,False,False,False,False
5,AMZN,232.789993,10.778526,25.501703,-14.723177,True,False,False,False,20,7.577645,5.115136e+07,False,False,False,False
9,TSLA,405.040009,6.354203,25.501703,-19.147500,False,False,False,False,0,3.323126,4.919301e+07,False,False,True,True
3,MSFT,367.179993,-4.088774,25.501703,-29.590477,False,False,False,False,0,13.611856,4.180171e+07,False,False,False,False
4,META,563.849976,-6.656627,25.501703,-32.158330,False,False,False,False,0,7.441700,1.794730e+07,False,False,False,False
7,NFLX,72.879997,-21.953310,25.501703,-47.455012,False,False,False,False,0,13.995616,5.636011e+07,False,False,False,False


In [95]:
def setup_score(daily, hourly, current_time):
    current_time_clean = pd.Timestamp(current_time).tz_localize(None)
    current_date = current_time_clean.normalize()

    daily_clean = daily.copy()
    daily_clean.index = daily_clean.index.tz_localize(None)

    daily_past = daily_clean.loc[:current_date].iloc[:-1]

    if len(daily_past) < 63:
        return 0

    stock_3m_return = (daily_past["Close"].iloc[-1] / daily_past["Close"].iloc[-63] - 1) * 100

    last_hour = hourly.loc[:current_time].iloc[-1]
    recent_10d_high = daily_past["High"].tail(10).max()
    distance_to_10d_high = ((recent_10d_high - last_hour["Close"]) / last_hour["Close"]) * 100

    score = stock_3m_return

    # reward being close to high, but not above it
    if 0 < distance_to_10d_high < 8:
        score += 10

    # reward strong hourly trend
    if last_hour["Close"] > last_hour["EMA20"]:
        score += 5

    if last_hour["Close"] > last_hour["EMA50"]:
        score += 5

    return score

In [96]:
def backtest_portfolio(tickers):
    all_trades = []

    for ticker in tickers:
        trades = backtest_hourly_strategy(ticker)

        if len(trades) > 0:
            trades["Score"] = trades.apply(
                lambda row: setup_score(
                    download_data(ticker)[0],
                    download_data(ticker)[1],
                    row["Signal Time"]
                ),
                axis=1
            )

            all_trades.append(trades)

    if len(all_trades) == 0:
        return pd.DataFrame()

    all_trades = pd.concat(all_trades, ignore_index=True)

    all_trades["Entry Date"] = pd.to_datetime(all_trades["Entry Time"]).dt.date

    best_trades = (
        all_trades
        .sort_values("Score", ascending=False)
        .groupby("Entry Date")
        .head(1)
        .reset_index(drop=True)
    )

    return best_trades

In [112]:
watchlist = [
    "AAPL", "MSFT", "NVDA", "AMD", "AVGO",
    "AMZN", "META", "GOOGL", "NFLX", "TSLA",

    "INTC", "QCOM", "CSCO", "ADBE", "AMAT",
    "LRCX", "KLAC", "ADI", "NXPI", "MU",

    "CRWD", "FTNT", "PANW", "ZS", "DDOG",
    "MDB", "SNOW", "PLTR", "TEAM", "DOCU",

    "INTU", "CDNS", "SNPS", "ADSK", "WDAY",
    "ROP", "PAYX", "CTAS", "ODFL", "FAST",

    "MELI", "ABNB", "BKNG", "SBUX", "COST",
    "TMUS", "MAR", "ISRG", "REGN", "GILD"
]

portfolio_trades = backtest_portfolio(watchlist)
portfolio_trades

AAPL trades: 0
MSFT trades: 0
NVDA trades: 1
AMD trades: 2
AVGO trades: 1
AMZN trades: 1
META trades: 0
GOOGL trades: 2
NFLX trades: 0
TSLA trades: 0
INTC trades: 0
QCOM trades: 2
CSCO trades: 1
ADBE trades: 0
AMAT trades: 3
LRCX trades: 2
KLAC trades: 1
ADI trades: 1
NXPI trades: 1
MU trades: 1
CRWD trades: 0
FTNT trades: 1
PANW trades: 1
ZS trades: 0
DDOG trades: 0
MDB trades: 0
SNOW trades: 0
PLTR trades: 0
TEAM trades: 0
DOCU trades: 0
INTU trades: 0
CDNS trades: 0
SNPS trades: 0
ADSK trades: 0
WDAY trades: 0
ROP trades: 0
PAYX trades: 0
CTAS trades: 0
ODFL trades: 1
FAST trades: 0
MELI trades: 0
ABNB trades: 1
BKNG trades: 0
SBUX trades: 1
COST trades: 0
TMUS trades: 0
MAR trades: 0
ISRG trades: 0
REGN trades: 0
GILD trades: 0


,Ticker,Signal Time,Entry Time,Exit Time,Entry Price,Exit Price,Initial Stop,Final Stop,R Result,Moved BE,Score,Entry Date
0,AMD,2026-06-17 12:30:00-04:00,2026-06-17 14:30:00-04:00,2026-06-17 15:30:00-04:00,525.400024,519.572103,515.208400,519.572103,-0.571834,False,174.331691,2026-06-17
1,AMD,2026-05-20 12:30:00-04:00,2026-05-20 14:30:00-04:00,2026-05-21 09:30:00-04:00,446.549988,436.235513,436.235513,436.235513,-1.000000,False,123.594433,2026-05-20
2,QCOM,2026-06-02 14:30:00-04:00,2026-06-03 09:30:00-04:00,2026-06-04 09:30:00-04:00,243.979996,244.998935,238.264422,244.998935,0.178274,True,85.778612,2026-06-03
3,FTNT,2026-06-11 10:30:00-04:00,2026-06-11 13:30:00-04:00,2026-06-12 09:30:00-04:00,144.919998,142.734235,142.734235,142.734235,-1.000000,False,84.549766,2026-06-11
4,MU,2026-05-20 14:30:00-04:00,2026-05-21 09:30:00-04:00,2026-06-04 09:30:00-04:00,734.000000,1043.689713,715.228307,1043.689713,16.497697,True,77.423021,2026-05-21
5,KLAC,2026-06-08 11:30:00-04:00,2026-06-09 09:30:00-04:00,2026-06-09 10:30:00-04:00,213.520996,213.520996,208.923623,213.520996,0.000000,True,54.994047,2026-06-09
6,GOOGL,2026-05-14 11:30:00-04:00,2026-05-14 13:30:00-04:00,2026-05-15 09:30:00-04:00,402.299805,398.937147,398.937147,398.937147,-1.000000,False,50.297733,2026-05-14
7,AMZN,2026-05-13 12:30:00-04:00,2026-05-13 15:30:00-04:00,2026-05-14 09:30:00-04:00,270.399597,268.587676,268.412958,268.587676,-0.912053,False,50.252844,2026-05-13
8,AVGO,2026-05-01 09:30:00-04:00,2026-05-01 13:30:00-04:00,2026-05-04 09:30:00-04:00,422.169891,417.430254,417.430254,417.430254,-1.000000,False,45.997588,2026-05-01
9,GOOGL,2026-05-27 09:30:00-04:00,2026-05-27 12:30:00-04:00,2026-05-27 15:30:00-04:00,392.839996,389.828113,389.828113,389.828113,-1.000000,False,44.282522,2026-05-27


In [113]:
backtest_stats(portfolio_trades)

{'Trades': 15,
 'Wins': 3,
 'Losses': 10,
 'Breakevens': 2,
 'Win Rate': 0.2,
 'Average R': np.float64(0.9039438073923386),
 'Total R': np.float64(13.55915711088508),
 'Best R': 16.49769736891026,
 'Worst R': -1.000000000000003}

In [101]:
def download_data(symbol):
    daily = yf.download(symbol, period="2y", interval="1d", auto_adjust=False, progress=False)
    hourly = yf.download(symbol, period="60d", interval="1h", auto_adjust=False, progress=False)

    if daily.empty or hourly.empty:
        raise ValueError(f"No data for {symbol}")

    daily = clean_columns(daily)
    hourly = clean_columns(hourly)

    if len(hourly) < 20 or len(daily) < 120:
        raise ValueError(f"Not enough data for {symbol}")

    daily = add_emas(daily)
    hourly = add_emas(hourly)
    hourly = add_atr(hourly)

    return daily, hourly

In [102]:
def backtest_portfolio(tickers):
    all_trades = []

    for ticker in tickers:
        try:
            trades = backtest_hourly_strategy(ticker)

            if len(trades) > 0:
                daily, hourly = download_data(ticker)
                trades["Score"] = [
                    setup_score(daily, hourly, row["Signal Time"])
                    for _, row in trades.iterrows()
                ]
                all_trades.append(trades)

            print(ticker, "trades:", len(trades))

        except Exception as e:
            print(ticker, "skipped:", e)

    if len(all_trades) == 0:
        return pd.DataFrame()

    all_trades = pd.concat(all_trades, ignore_index=True)
    all_trades["Entry Date"] = pd.to_datetime(all_trades["Entry Time"], utc=True).dt.date

    return (
        all_trades
        .sort_values("Score", ascending=False)
        .groupby("Entry Date")
        .head(1)
        .reset_index(drop=True)
    )

In [114]:
baseline_stats = backtest_stats(portfolio_trades)
baseline_stats

{'Trades': 15,
 'Wins': 3,
 'Losses': 10,
 'Breakevens': 2,
 'Win Rate': 0.2,
 'Average R': np.float64(0.9039438073923386),
 'Total R': np.float64(13.55915711088508),
 'Best R': 16.49769736891026,
 'Worst R': -1.000000000000003}

In [117]:
system_name = "v1_baseline"

portfolio_trades.to_csv(f"results/{system_name}_trades.csv", index=False)

pd.DataFrame([baseline_stats]).to_csv(
    f"results/{system_name}_stats.csv",
    index=False
)

In [116]:
import os

os.makedirs("results", exist_ok=True)